# Speech-to-Text Demo

## 1. Setup

In [ ]:
# Install missing packages
!pip install -q -U transformers accelerate pydub torchaudio gdown jiwer

In [ ]:
import os
import io
import json
import wave
import torch
import jiwer
import gdown
import shutil
import zipfile
import torchaudio
import numpy as np
from base64 import b64decode
from pydub import AudioSegment
from transformers import pipeline
from google.colab.output import eval_js
from IPython.display import Javascript, Audio, display

In [ ]:
_RECORD_JS = """
async function record(ms, prompt) {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });

  const div = document.createElement('div');
  div.innerHTML = `<p>Click, then <b>${prompt}</b> (auto-stops after ${ms / 1000}s)</p><button>🎤 Start</button>`;
  document.body.appendChild(div);
  const btn = div.querySelector('button');
  await new Promise(resolve => { btn.onclick = resolve; });
  btn.textContent = `● Recording... (${ms / 1000}s)`;

  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  const stopped = new Promise(resolve => { recorder.onstop = resolve; });
  recorder.start();
  setTimeout(() => recorder.stop(), ms);
  await stopped;
  stream.getTracks().forEach(t => t.stop());
  div.remove();

  const reader = new FileReader();
  reader.readAsDataURL(new Blob(chunks));
  return new Promise(resolve => { reader.onloadend = () => resolve(reader.result); });
}
"""

def _to_waveform(segment):
    segment = segment.set_channels(1)
    waveform = np.array(segment.get_array_of_samples()).astype(np.float32)
    waveform /= 1 << (8 * segment.sample_width - 1)
    return waveform, segment.frame_rate

def record_audio(seconds=8, prompt="speak now"):
    """Record from the browser mic, returns (waveform float32 numpy, sample_rate)."""
    display(Javascript(_RECORD_JS))
    data_url = eval_js(f"record({seconds * 1000}, {json.dumps(prompt)})")
    raw = b64decode(data_url.split(",", 1)[1])
    return _to_waveform(AudioSegment.from_file(io.BytesIO(raw)))

my_recordings = {}
my_transcripts = {}

def save_recording(name, waveform, sr, text=None):
    """Keep a recording (and its known text, if given) in memory for this session - not written to disk or Drive, gone on runtime reset."""
    my_recordings[name] = (waveform, sr)
    if text is not None:
        my_transcripts[name] = text

def clear_recordings():
    """Delete every in-session recording (and its text) from memory - no undo."""
    my_recordings.clear()
    my_transcripts.clear()

def download_recordings():
    """Zip every recording + a transcripts.json (same format as the common bank) and trigger a browser download to your PC."""
    from google.colab import files

    zip_path = "/content/my_recordings.zip"
    with zipfile.ZipFile(zip_path, "w") as zf:
        for name, (waveform, sr) in my_recordings.items():
            pcm16 = np.clip(waveform * 32768, -32768, 32767).astype(np.int16)
            wav_path = f"/content/{name}.wav"
            with wave.open(wav_path, "wb") as wav_file:
                wav_file.setnchannels(1)
                wav_file.setsampwidth(2)
                wav_file.setframerate(sr)
                wav_file.writeframes(pcm16.tobytes())
            zf.write(wav_path, arcname=f"{name}.wav")
        zf.writestr("transcripts.json", json.dumps(my_transcripts, ensure_ascii=False, indent=2))
    files.download(zip_path)

def load_sample(name):
    """Load a clip by name - checks your in-session recordings (Section 3) first, then the common bank (Section 2)."""
    if name in my_recordings:
        return my_recordings[name]
    path = f"{AUDIO_SAMPLES_DIR}/{name}"
    if os.path.exists(path):
        return _to_waveform(AudioSegment.from_file(path))
    raise FileNotFoundError(f"'{name}' not found in your recordings or {AUDIO_SAMPLES_DIR}")

def to_16k(waveform, sr):
    """Whisper expects 16kHz audio."""
    if sr == 16000:
        return waveform
    resampled = torchaudio.functional.resample(
        torch.from_numpy(waveform), orig_freq=sr, new_freq=16000
    )
    return resampled.numpy()

# WER/CER, case-insensitive with every non-letter/non-digit character replaced by a space
_TEXT_NORMALIZE = jiwer.SubstituteRegexes({r"[^\w\s]": " "})

_WER_TRANSFORM = jiwer.Compose([
    jiwer.ToLowerCase(),
    _TEXT_NORMALIZE,
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(),
    jiwer.ReduceToListOfListOfWords(),
])
_CER_TRANSFORM = jiwer.Compose([
    jiwer.ToLowerCase(),
    _TEXT_NORMALIZE,
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(),
    jiwer.ReduceToListOfListOfChars(),
])

def word_error_rates(reference, hypothesis):
    """(WER, CER) between a reference transcript and a model hypothesis."""
    wer = jiwer.wer(reference, hypothesis, reference_transform=_WER_TRANSFORM, hypothesis_transform=_WER_TRANSFORM)
    cer = jiwer.cer(reference, hypothesis, reference_transform=_CER_TRANSFORM, hypothesis_transform=_CER_TRANSFORM)
    return wer, cer

## 2. Load audio samples

In [ ]:
# audio samples: a read-only folder of example clips
AUDIO_SAMPLES_DIR = "/content/audio_samples"
AUDIO_SAMPLES_FOLDER_ID = "1YgInfep4vnRA1pX4-7h8e4G4AxqzPLg6"

In [ ]:
shutil.rmtree(AUDIO_SAMPLES_DIR, ignore_errors=True)
os.makedirs(AUDIO_SAMPLES_DIR, exist_ok=True)

try:

    # Listing the folder is one lightweight request (not a download); find
    # whichever .zip is in there right now instead of hardcoding a file id.
    listing = gdown.download_folder(id=AUDIO_SAMPLES_FOLDER_ID, skip_download=True, use_cookies=False)
    zip_entries = [f for f in listing if f.path.endswith(".zip")]
    if len(zip_entries) != 1:
        print(f"Expected exactly one .zip in the folder, found {len(zip_entries)}: {[f.path for f in zip_entries]} - using the first one.")
    zip_path = gdown.download(id=zip_entries[0].id, output="/content/audio_samples.zip", quiet=False)

    with zipfile.ZipFile(zip_path) as zf:
        # Flatten structure - extract every file to AUDIO_SAMPLES_DIR directly
        for member in zf.namelist():
            if member.endswith("/"):
                continue
            with zf.open(member) as src, open(f"{AUDIO_SAMPLES_DIR}/{os.path.basename(member)}", "wb") as dst:
                dst.write(src.read())
except Exception as e:
    print(f"Couldn't download the common sample bank ({e!r}) — you can still use Section 3 to record your own samples.")

audio_samples = sorted(f for f in os.listdir(AUDIO_SAMPLES_DIR) if f != "transcripts.json")

transcripts_path = f"{AUDIO_SAMPLES_DIR}/transcripts.json"
if os.path.exists(transcripts_path):
    with open(transcripts_path) as f:
        transcripts = json.load(f)
else:
    transcripts = {}

print(f"\n{audio_samples}")
print(f"\n{len(audio_samples)} audio sample(s) loaded.")
print(f"\n{len(transcripts)} reference transcript(s) loaded.")

In [ ]:
# Preview all samples.
for name in audio_samples:
    print(f"\n{name}\n{transcripts[name]}")
    display(Audio(f"{AUDIO_SAMPLES_DIR}/{name}"))

## 3. Create new samples (yours only)
- set sample name
- set text so we can calculate WER/CER later
- run the cell, it will fail until you allow mic and run it again
- click start and speak, the duration of the recording is fixed by the function argument

In [ ]:
# sample_name = "no_speech"
# text = " "

# waveform, sr = record_audio(8, prompt=f'read: "{text}"')
# save_recording(sample_name, waveform, sr, text=text)
# display(Audio(waveform, rate=sr))

Preview everything you've recorded so far.

In [ ]:
for name, (waveform, sr) in my_recordings.items():
    print(f"{name}: {my_transcripts.get(name, '(no reference text saved)')}")
    display(Audio(waveform, rate=sr))

Download recordings if you want to keep them after the runtime resets.

In [ ]:
# download_recordings()

Delete current recordings to start over.

In [ ]:
# clear_recordings()

## 4. Load models

We will compare two ASR models:

- **[`openai/whisper-large-v3`](https://huggingface.co/openai/whisper-large-v3-turbo)**: the original, multilingual Whisper checkpoint (handles Slovak, English, and 90+ other languages, but Slovak is a small slice of its training data).
- **[`kinit/whisper-large-v3-sk`](https://huggingface.co/kinit/whisper-large-v3-sk)**: KInIT's version of the model fine-tuned specifically on Slovak. Performance on other languages is radically degraded.

In [ ]:
MODELS = {
    "whisper-large-v3": "openai/whisper-large-v3",
    "whisper-large-v3-sk": "kinit/whisper-large-v3-sk",
}

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device.startswith("cuda") else torch.float32
pipeline_device = 0 if device.startswith("cuda") else -1
print(device)

asr_pipelines = {
    name: pipeline(
        "automatic-speech-recognition",
        model=model_id,
        torch_dtype=dtype,
        device=pipeline_device,
    )
    for name, model_id in MODELS.items()
}

## 5. Transcribe: pick a sample & model

`transcribe(sample_name, model_name, language=None, task="transcribe", reference_text=None)` ties everything together:

- `sample_name`: a filename from the common samples (e.g. `"sk_female_01.wav"`) or a name you gave in `save_recording()` (e.g. `"example_01"`, no extension).
- `model_name`: a key from `MODELS` (Section 4): `"whisper-large-v3"` or `"whisper-large-v3-sk"`.
- `language`: the spoken language of the sample (e.g. `"slovak"`, `"english"`). Recommended for the Slovak fine-tune; leave as `None` to let the original model auto-detect.
- `task`: `"transcribe"` (stays in the spoken language) or `"translate"` (converts to English — only the original multilingual model supports this).
- `reference_text`: the known-correct transcript, to print WER/CER alongside the result. Auto-filled by filename/name from `transcripts.json` or `my_transcripts` (populated by `save_recording(..., text=...)`). Pass it explicitly only to override.

`compare(sample_name, ...)` takes the same arguments and runs every model in `MODELS` on the same sample, so the WER/CER contrast is visible directly instead of split across separate calls.

In [ ]:
def transcribe(sample_name, model_name, language=None, task="transcribe", reference_text=None):
    print(f"\n--- {model_name} ---")
    waveform, sr = load_sample(sample_name)
    display(Audio(waveform, rate=sr))

    generate_kwargs = {"task": task}
    if language:
        generate_kwargs["language"] = language

    result = asr_pipelines[model_name](
        {"array": to_16k(waveform, sr), "sampling_rate": 16000},
        generate_kwargs=generate_kwargs,
    )
    hypothesis = result["text"]
    label = "Transcribed" if task == "transcribe" else "Translated to English"
    print(f"{label}: {hypothesis}")

    # WER/CER only makes sense when comparing against a reference in the same language
    reference = reference_text if reference_text is not None else (my_transcripts.get(sample_name) or transcripts.get(sample_name))
    if reference and task == "transcribe":
        wer, cer = word_error_rates(reference, hypothesis)
        print(f"Reference:  {reference}")
        print(f"WER: {wer:.1%}   CER: {cer:.1%}")


def compare(sample_name, language=None, task="transcribe", reference_text=None):
    """Run every model in MODELS on the same sample, back to back."""
    for model_name in MODELS:
        transcribe(sample_name, model_name, language=language, task=task, reference_text=reference_text)

### Clean English audio
Usually no errors. This is where the models are already almost flawless.

In [ ]:
transcribe("en_librispeech_01.wav", "whisper-large-v3")
transcribe("en_librispeech_02.wav", "whisper-large-v3")

### Clean Multilingual audio on big languages
Also works very well. It is common for ASR models to support also translation. Whisper supports translation to English.

In [ ]:
transcribe("de_fleurs.wav", "whisper-large-v3")
transcribe("es_fleurs.wav", "whisper-large-v3")
transcribe("fr_fleurs.wav", "whisper-large-v3")

In [ ]:
transcribe("de_fleurs.wav", "whisper-large-v3", task="translate")
transcribe("es_fleurs.wav", "whisper-large-v3", task="translate")
transcribe("fr_fleurs.wav", "whisper-large-v3", task="translate")

In [ ]:
# Chinese is the second biggest language after English
# but WER is not a good metric for this language
transcribe("zh_fleurs_01.wav", "whisper-large-v3")
transcribe("zh_fleurs_01.wav", "whisper-large-v3", task="translate")

transcribe("zh_fleurs_02.wav", "whisper-large-v3")
transcribe("zh_fleurs_02.wav", "whisper-large-v3", task="translate")

### Other smaller languages
Error rate increases for languages less represented in the training data

In [ ]:
# Polish
transcribe("pl_fleurs.wav", "whisper-large-v3")
transcribe("pl_fleurs.wav", "whisper-large-v3", task="translate")

In [ ]:
# Hungarian
transcribe("hu_fleurs.wav", "whisper-large-v3")
transcribe("hu_fleurs.wav", "whisper-large-v3", task="translate")

In [ ]:
# Lao - smallest language from the original Whisper training data
transcribe("lo_fleurs.wav", "whisper-large-v3")
transcribe("lo_fleurs.wav", "whisper-large-v3", task="translate")

### Slovak audio samples
Slovak is a low-resource language but with clean audio, it work ok. We compare it with Whisper fine-tuned only for Slovak.

Speech for children is especially difficult since it is almost non-existent in train data.

In [ ]:
compare("sk_female_01.wav", language="slovak")

In [ ]:
compare("sk_female_02.wav", language="slovak")

In [ ]:
compare("sk_girl_01.wav", language="slovak")

In [ ]:
compare("sk_girl_02.wav", language="slovak")

In [ ]:
compare("sk_girl_03.wav", language="slovak")

In [ ]:
compare("sk_small_child_01.wav", language="slovak")

In [ ]:
compare("sk_small_child_02.wav", language="slovak")

### Silence can cause hallucinations
This is a known limitation of Whisper and is mitigated in more recent models.

In [ ]:
transcribe("silence_pure.wav", "whisper-large-v3")
transcribe("silence_room_tone.wav", "whisper-large-v3")
transcribe("silence_hum.wav", "whisper-large-v3")
transcribe("silence_long_pause.wav", "whisper-large-v3")
transcribe("silence_breathing.wav", "whisper-large-v3")

In [ ]:
transcribe("silence_pure.wav", "whisper-large-v3-sk")
transcribe("silence_room_tone.wav", "whisper-large-v3-sk")
transcribe("silence_hum.wav", "whisper-large-v3-sk")
transcribe("silence_long_pause.wav", "whisper-large-v3-sk")
transcribe("silence_breathing.wav", "whisper-large-v3-sk")

Hallucinations can be even worse if we fix the language.

In [ ]:
transcribe("silence_pure.wav", "whisper-large-v3", language="sk")
transcribe("silence_room_tone.wav", "whisper-large-v3", language="sk")
transcribe("silence_hum.wav", "whisper-large-v3", language="sk")
transcribe("silence_long_pause.wav", "whisper-large-v3", language="sk")
transcribe("silence_breathing.wav", "whisper-large-v3", language="sk")

In [ ]:
transcribe("no_speech.wav", "whisper-large-v3")
transcribe("no_speech.wav", "whisper-large-v3", language="sk")

transcribe("no_speech.wav", "whisper-large-v3-sk")

### Catastrophic forgetting is extreme when fine-tuning on single language
Our experiments show that LoRA can improve this a lot but languages closest to Slovak are still much worse after fine-tuning.

In [ ]:
compare("en_librispeech_01.wav")
compare("en_librispeech_02.wav")

In [ ]:
# compare("es_fleurs.wav")
compare("fr_fleurs.wav")
# compare("de_fleurs.wav")
# compare("zh_fleurs_01.wav")
compare("zh_fleurs_02.wav")
compare("hu_fleurs.wav")
compare("pl_fleurs.wav")